# Iteration 2 · AUM-only continuation at 40% noise

Run in a Google Colab GPU runtime. This notebook imports the verified epoch-6 warm-up, then runs six adaptive continuation epochs using the within-class rank of negative AUM. It uses a pinned read-only checkout of the original source and writes only to this experiment's own output directory. Use only the checkpoint supplied by your teammate, whose expected digest is pinned below.

Run the setup and verification cells first. Then run training and evaluation separately. To resume after a disconnect, rerun the cells using the same output folder and software environment. The official test set is used only by the final evaluation cell.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile

from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/ido-aum-iteration-2')
inputs = root / 'inputs'
outputs = root / 'outputs' / 'noise_40_aum_only'
repo = Path('/content/ido-aum-lite-pinned')
revision = '695f50fef1915f0ab2d370ba6c8cda1282e9dae2'

def git(*args):
    return subprocess.check_output(['git', *args], cwd=repo, text=True).strip()

if not repo.exists():
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/Avilas01/ido-aum-lite.git', str(repo)], check=True)
    git('checkout', '--detach', revision)
if git('rev-parse', 'HEAD') != revision or git('status', '--porcelain'):
    raise RuntimeError('Expected the clean, pinned upstream checkout')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'], check=True)
sys.path.insert(0, str(repo))
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab, then rerun setup')
device = torch.device('cuda')
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(device))


In [ ]:
from config import configuration_fingerprint
from src.checkpoints import (CheckpointStore, atomic_json, capture_rng, code_identity,
                             environment, file_digest, model_state_hash, restore_rng,
                             seed_all, seed_streams)
from src.data import (TrainingData, _cifar10, array_checksum, load_training_metadata,
                      make_training_datasets)
from src.evaluate import TestEvaluator, cached_accuracy
from src.model import build_model
from src.scores import suspicion_scores
from src.statistics import SampleStatistics, collect_epoch_statistics
from src.train import _generators, _loaders, _validate_state, train_epoch

from time import perf_counter
import tempfile

expected_checkpoint = '89c78c575078b985e8ada977ea913300bfd98dd0ac57fdc196e6d3f011e438b9'
run = json.loads((inputs / 'run.json').read_text(encoding='utf-8-sig'))
original = run['identity']
settings = original['configuration']
streams = seed_streams(settings['seed'])
if (original['code'] != code_identity(repo)
        or original['configuration_sha256'] != configuration_fingerprint(settings)
        or original['dataset']['noise_ratio'] != 0.4
        or original['schedule'] != {'warmup_epochs': 6, 'continuation_epochs': 6}
        or run['seed_streams'] != streams):
    raise ValueError('Original source, configuration, schedule or seed mismatch')

warmup_identity = {**original, 'stage': 'warmup', 'target_epochs': 6}
with tempfile.TemporaryDirectory(dir='/content') as temp:
    with ZipFile(inputs / 'warmup-20261125T210041Z-1-001.zip') as archive:
        journal = json.loads(archive.read('warmup/latest.json'))
        entries = [item for item in journal['entries']
                   if item['epoch'] == 6 and item['sha256'] == expected_checkpoint]
        if journal['identity'] != warmup_identity or len(entries) != 1:
            raise ValueError('Expected the published 40% epoch-6 checkpoint')
        entry = entries[0]
        if Path(entry['file']).name != entry['file']:
            raise ValueError('Unexpected checkpoint filename')
        checkpoint = Path(temp) / 'warmup.pt'
        with archive.open('warmup/' + entry['file']) as source, checkpoint.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        if checkpoint.stat().st_size != entry['size_bytes'] or file_digest(checkpoint) != expected_checkpoint:
            raise ValueError('Warm-up checkpoint checksum mismatch')
        shared = torch.load(checkpoint, map_location='cpu', weights_only=False)

    with ZipFile(inputs / 'data-20261125T210041Z-1-001.zip') as archive:
        for name in ('metadata.json', 'training.npz'):
            (Path(temp) / name).write_bytes(archive.read('data/' + name))
    metadata = load_training_metadata(temp)

tracker = _validate_state(shared, warmup_identity, 'warmup', 6, settings['train_size'], 6)
if shared['completed_epoch'] != 6 or any(
    original['dataset'].get(key) != value for key, value in metadata.summary().items()
):
    raise ValueError('Warm-up and saved training metadata do not match')

scores = suspicion_scores(tracker.we, tracker.aum, metadata.noisy_labels,
                          observation_counts=tracker.observation_counts)['aum']
print(f'Verified epoch 6 and {scores.size:,} aligned AUM-only starting scores.')
runtime = environment(device)
if runtime != run['environment']:
    print('Runtime differs from Iteration 1. The actual runtime will be recorded in results.')


In [ ]:
# Runs only the new AUM-only branch. Existing checkpoints in this output folder
# allow epoch-boundary resume. Use a new output folder for an independent run.
data_root = Path('/content/ido-aum-cifar10')
archive_root = root / 'cache' / 'cifar10'
branch_identity = {
    'method': 'aum_only',
    'protocol_version': 1,
    'warmup_sha256': expected_checkpoint,
    'original_run': original,
    'runtime': runtime,
}
store = CheckpointStore(outputs / 'continuation', branch_identity)
saved, final_entry = store.load() if store.exists() else (None, None)
state = saved
if saved is not None:
    epoch_count = saved['completed_epoch']
    if (not 1 <= epoch_count <= 6 or saved['initial_model_sha256'] != shared['model_sha256']
            or saved['model_sha256'] != model_state_hash(saved['model'])
            or [row['epoch'] for row in saved['logs']] != list(range(1, epoch_count + 1))):
        raise ValueError('Invalid AUM-only resume checkpoint')

if saved is None or saved['completed_epoch'] < 6:
    cifar = _cifar10(data_root, train=True, download=True, archive_root=archive_root)
    images = cifar.data[metadata.original_indices]
    if array_checksum(images) != original['dataset']['images_sha256']:
        raise ValueError('CIFAR training image checksum mismatch')
    datasets = make_training_datasets(TrainingData(images, metadata))

    seed_all(streams['model'])
    model = build_model(settings).to(device)
    model.load_state_dict(shared['model'])
    if model_state_hash(model.state_dict()) != shared['model_sha256']:
        raise ValueError('Model did not start from the complete warm-up state')
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'],
                                  weight_decay=settings['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=settings['amp'])
    generators = _generators(streams)
    training, statistics = _loaders(datasets, settings, generators, adaptive=True)

    if saved is None:
        tracker = SampleStatistics.from_state_dict(shared['statistics'])
        logs = []
        seed_all(streams['branch'])
    else:
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        tracker = SampleStatistics.from_state_dict(saved['statistics'])
        logs = list(saved['logs'])
        if tracker.tracked_epochs != 6 + len(logs):
            raise ValueError('Resume tracker and continuation epoch mismatch')
        restore_rng(saved['rng'], generators)

    for epoch in range(len(logs) + 1, 7):
        torch.cuda.synchronize(device)
        started = perf_counter()
        scores = suspicion_scores(tracker.we, tracker.aum, metadata.noisy_labels,
                                  observation_counts=tracker.observation_counts)['aum']
        diagnostics = train_epoch(model, training, optimizer, scaler, settings, device, scores=scores)
        common = collect_epoch_statistics(model, statistics, device=device, tracker=tracker,
                                          epoch=tracker.tracked_epochs + 1)
        torch.cuda.synchronize(device)
        logs.append({'epoch': epoch, **diagnostics, 'common_noisy_ce': common['noisy_ce'],
                     'epoch_seconds_excluding_checkpoint': perf_counter() - started})
        state = {
            'schema_version': 1, 'identity': branch_identity, 'stage': 'aum_only',
            'completed_epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict(), 'statistics': tracker.state_dict(),
            'rng': capture_rng(generators), 'logs': logs,
            'initial_model_sha256': shared['model_sha256'],
            'model_sha256': model_state_hash(model.state_dict()),
        }
        final_entry = store.save(state)
        print(f"Continuation epoch {epoch}/6: CE={common['noisy_ce']:.6f}", flush=True)

atomic_json(outputs / 'epochs.json', state['logs'])
print('AUM-only continuation complete.')


In [ ]:
if state['completed_epoch'] != 6:
    raise ValueError('Complete all six continuation epochs before testing')

evaluator = TestEvaluator(data_root, archive_root, device, runtime)
accuracy, cache_path = cached_accuracy(state, final_entry, settings, evaluator, outputs)
comparison = {
    'method': 'aum_only',
    'noise_ratio': 0.4,
    'seed': settings['seed'],
    'warmup_checkpoint_sha256': expected_checkpoint,
    'final_checkpoint_sha256': final_entry['sha256'],
    'accuracy': accuracy,
    'accuracy_cache': cache_path,
    'amp_skipped_steps': sum(row['amp_skipped_steps'] for row in state['logs']),
    'iteration_1_test_accuracy_pct': {'standard': 63.80, 'we_only': 68.61, 'hybrid': 68.48},
    'delta_vs_we_only_pp': round(accuracy['accuracy_pct'] - 68.61, 4),
    'delta_vs_hybrid_pp': round(accuracy['accuracy_pct'] - 68.48, 4),
    'runtime': runtime,
    'original_runtime': run['environment'],
}
atomic_json(outputs / 'comparison.json', comparison)
print(json.dumps(comparison, indent=2))
print(f'Results saved to {outputs / "comparison.json"}')
